[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

Welcome to LangChain Academy! 

## Context

At LangChain, we aim to make it easy to build LLM applications. One type of LLM application you can build is an agent. There’s a lot of excitement around building agents because they can automate a wide range of tasks that were previously impossible. 

In practice though, it is incredibly difficult to build systems that reliably execute on these tasks. As we’ve worked with our users to put agents into production, we’ve learned that more control is often necessary. You might need an agent to always call a specific tool first or use different prompts based on its state. 

To tackle this problem, we’ve built [LangGraph](https://langchain-ai.github.io/langgraph/) — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to help developers add better precision and control into agent workflows, suitable for the complexity of real-world systems.

## Course Structure

The course is structured as a set of modules, with each module focused on a particular theme related to LangGraph. You will see a folder for each module, which contains a series of notebooks. A video will accompany each notebook to help walk through the concepts, but the notebooks are also stand-alone, meaning that they contain explanations and can be viewed independently of the videos. Each module folder also contains a `studio` folder, which contains a set of graphs that can be loaded into [LangGraph Studio](https://github.com/langchain-ai/langgraph-studio), our IDE for building LangGraph applications.

## Setup

Before you begin, please follow the instructions in the `README` to create an environment and install dependencies.

## Chat models

In this course, we'll be using [Chat Models](https://python.langchain.com/v0.2/docs/concepts/#chat-models), which do a few things take a sequence of messages as inputs and return chat messages as outputs. LangChain does not host any Chat Models, rather we rely on third party integrations. [Here](https://python.langchain.com/v0.2/docs/integrations/chat/) is a list of 3rd party chat model integrations within LangChain! By default, the course will use [ChatOpenAI](https://python.langchain.com/v0.2/docs/integrations/chat/openai/) because it is both popular and performant. As noted, please ensure that you have an `OPENAI_API_KEY`.

Let's check that your `OPENAI_API_KEY` is set and, if not, you will be asked to enter it.

In [18]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_groq langchain_core langchain_community tavily-python

In [8]:
# import os, getpass

# def _set_env(var: str):
#     if not os.environ.get(var):
#         os.environ[var] = getpass.getpass(f"{var}: ")

# # This will prompt you securely if not already set
# _set_env("OPENAI_API_KEY")
# _set_env("GROQ_API_KEY")

# # Now you can use the keys
# print("Keys are set!")


In [16]:
import os
import getpass
from pathlib import Path

def load_env_file(env_file=".env"):
    """Load environment variables from a .env file"""
    env_path = Path(env_file)
    
    if env_path.exists():
        print(f"Loading environment variables from {env_file}")
        with open(env_path, 'r') as file:
            for line in file:
                line = line.strip()
                # Skip empty lines and comments
                if line and not line.startswith('#'):
                    # Split on first '=' only
                    if '=' in line:
                        key, value = line.split('=', 1)
                        # Remove quotes if present
                        value = value.strip('"\'')
                        os.environ[key.strip()] = value
        print("✅ Environment variables loaded from .env file")
    else:
        print(f"⚠️  No {env_file} file found")

def _set_env(var: str):
    """Set environment variable, prompting securely if not already set"""
    if not os.environ.get(var):
        print(f"🔑 {var} not found in environment")
        os.environ[var] = getpass.getpass(f"Enter {var}: ")
        print(f"✅ {var} has been set")
    else:
        print(f"✅ {var} already configured")

def setup_environment(env_vars=None, env_file=".env"):
    """
    Complete environment setup:
    1. Load from .env file if it exists
    2. Prompt for any missing required variables
    
    Args:
        env_vars: List of required environment variable names
        env_file: Path to .env file (default: ".env")
    """
    if env_vars is None:
        env_vars = ["GROQ_API_KEY", "OPENAI_API_KEY", "LANGSMITH_API_KEY", "TAVILY_API_KEY"]
    
    print("🚀 Setting up environment variables...")
    
    # First, try to load from .env file
    load_env_file(env_file)
    
    # Then check/prompt for required variables
    print("\n📋 Checking required environment variables:")
    for var in env_vars:
        _set_env(var)
    
    print("\n🎉 Environment setup complete!")

# Example usage:
if __name__ == "__main__":
    # Setup with default variables
    # setup_environment(env_file="../.env")
    
    # Or specify your required variables
    setup_environment(["LANGSMITH_API_KEY", "GROQ_API_KEY", "TAVILY_API_KEY"], env_file="../.env")
    
    # Or use a different .env file
    # setup_environment(env_file="production.env")


🚀 Setting up environment variables...
Loading environment variables from ../.env
✅ Environment variables loaded from .env file

📋 Checking required environment variables:
✅ LANGSMITH_API_KEY already configured
✅ GROQ_API_KEY already configured
✅ TAVILY_API_KEY already configured

🎉 Environment setup complete!


[Here](https://python.langchain.com/v0.2/docs/how_to/#chat-models) is a useful how-to for all the things that you can do with chat models, but we'll show a few highlights below. If you've run `pip install -r requirements.txt` as noted in the README, then you've installed the `langchain-openai` package. With this, we can instantiate our `ChatOpenAI` model object. If you are signing up for the API for the first time, you should receive [free credits](https://community.openai.com/t/understanding-api-limits-and-free-tier/498517) that can be applied to any of the models. You can see pricing for various models [here](https://openai.com/api/pricing/). The notebooks will default to `gpt-4o` because it's a good balance of quality, price, and speed [see more here](https://help.openai.com/en/articles/7102672-how-can-i-access-gpt-4-gpt-4-turbo-gpt-4o-and-gpt-4o-mini), but you can also opt for the lower priced `gpt-3.5` series models. 

There are [a few standard parameters](https://python.langchain.com/v0.2/docs/concepts/#chat-models) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `temperature`: the sampling temperature

`Temperature` controls the randomness or creativity of the model's output where low temperature (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses. High temperature (close to 1) is good for creative tasks or generating varied responses. 

In [39]:
# from langchain_openai import ChatOpenAI
# gpt4o_chat = ChatOpenAI(model="gpt-4o", temperature=0)
gpt35_chat = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

In [19]:
# Free Groq models instead of OpenAI
from langchain_groq import ChatGroq

# Groq's powerful models (free tier)
mixtral_chat = ChatGroq(model="mixtral-8x7b-32768", temperature=0)
llama_chat = ChatGroq(model="llama2-70b-4096", temperature=0)

# Alternative models available on Groq
gemma_chat = ChatGroq(model="gemma-7b-it", temperature=0)
llama3_chat = ChatGroq(model="llama3-8b-8192", temperature=0)  # If available

In [20]:
# All free Groq models you can use:
models = [
    "mixtral-8x7b-32768",      # Mixtral - excellent performance
    "llama2-70b-4096",         # Llama 2 70B - very capable  
    "llama2-7b-2048",          # Smaller, faster Llama 2
    "gemma-7b-it",             # Google Gemma - instruction tuned
    "llama3-8b-8192",          # Llama 3 (if available)
    "llama3-70b-8192"          # Llama 3 70B (if available)
]

Chat models in LangChain have a number of [default methods](https://python.langchain.com/v0.2/docs/concepts/#runnable-interface). For the most part, we'll be using:

* `stream`: stream back chunks of the response
* `invoke`: call the chain on an input

And, as mentioned, chat models take [messages](https://python.langchain.com/v0.2/docs/concepts/#messages) as input. Messages have a role (that describes who is saying the message) and a content property. We'll be talking a lot more about this later, but here let's just show the basics.

In [41]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Lance")

# Message list
messages = [msg]

# Invoke the model with a list of messages 
# mixtral_chat.invoke(messages) # this model has been decomissioned
# llama_chat.invoke(messages)
# gemma_chat.invoke(messages)

# Most capable model (GPT-4 equivalent)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm2 = ChatGroq(model="qwen/qwen3-32b", temperature=0)


# Test it
response = llm.invoke(messages)
print(response.content)

Hello. How can I assist you today?


We get an `AIMessage` response. Also, note that we can just invoke a chat model with a string. When a string is passed in as input, it is converted to a `HumanMessage` and then passed to the underlying model.


In [32]:
llm.invoke("hello world")

AIMessage(content="Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 37, 'total_tokens': 62, 'completion_time': 0.055601053, 'prompt_time': 0.00180598, 'queue_time': 0.095440483, 'total_time': 0.057407033}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_9e1e8f8435', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--7f8144c1-7aac-4de0-a07c-2d2ef910a751-0', usage_metadata={'input_tokens': 37, 'output_tokens': 25, 'total_tokens': 62})

In [33]:
llm2.invoke("hello world")

AIMessage(content='<think>\nOkay, the user wrote "hello world". That\'s a classic first program in many programming languages. I should respond in a friendly and welcoming way. Maybe acknowledge their message and offer help if they need anything else. Keep it simple and positive. Let me make sure there\'s no hidden request here. They might just be testing or starting out. Alright, a straightforward reply should work.\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 10, 'total_tokens': 103, 'completion_time': 0.195948707, 'prompt_time': 0.000258256, 'queue_time': 0.089970951, 'total_time': 0.196206963}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--dd794fe4-e2e6-4a3c-9070-562852608bd7-0', usage_metadata={'input_tokens': 10, 'output_tokens': 93, 'total_tokens': 103})

The interface is consistent across all chat models and models are typically initialized once at the start up each notebooks. 

So, you can easily switch between models without changing the downstream code if you have strong preference for another provider.


## Search Tools

You'll also see [Tavily](https://tavily.com/) in the README, which is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier. Some lessons (in Module 4) will use Tavily by default but, of course, other search tools can be used if you want to modify the code for yourself.

In [34]:
_set_env("TAVILY_API_KEY")

✅ TAVILY_API_KEY already configured


In [37]:
from langchain_community.tools.tavily_search import TavilySearchResults
tavily_search = TavilySearchResults(max_results=5)
search_docs = tavily_search.invoke("What is LangGraph?")

In [38]:
search_docs

[{'title': 'LangGraph - Overview - Docs by LangChain',
  'url': 'https://docs.langchain.com/oss/python/langgraph/overview',
  'content': 'Trusted by companies shaping the future of agents - including Klarna, Replit, Elastic, and more - LangGraph is a low-level orchestration framework for building, managing, and deploying long-running, stateful agents.',
  'score': 0.94785136},
 {'title': 'What is LangGraph? - Analytics Vidhya',
  'url': 'https://www.analyticsvidhya.com/blog/2024/07/langgraph-revolutionizing-ai-agent/',
  'content': 'To sum up, LangGraph is a major advancement in the development of AI agents. It enables developers to push the limits of what’s possible with AI agents by eliminating the shortcomings of earlier systems and offering a flexible, graph-based framework for agent construction and execution. LangGraph is positioned to influence the direction of artificial intelligence significantly in the future. [...] LangGraph is a library built on top of Langchain that is des